# Figura: 4 representaciones del pipeline

Ejecuta las celdas en orden. Genera `figura_representaciones.png` lista para copiar a `Latex/imagenes/`.

In [ ]:
# CELDA 1 — instalar dependencias con versiones compatibles
!pip install plotly==5.18.0 kaleido==0.2.1 trimesh -q
print('OK')

In [ ]:
# CELDA 2 — montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELDA 3 — cargar malla y generar las 4 representaciones
import numpy as np
import trimesh

PLY = '/content/drive/MyDrive/Datos_E2_E3/General/NO_Objaverse_limpias_v2/0166cd3012284d0cb89d0c6548f9680c.ply'

# Cargar y normalizar a [-1, 1]
mesh = trimesh.load(PLY, force='mesh')
mesh.apply_translation(-mesh.centroid)
mesh.apply_scale(1.0 / mesh.scale)
v, f = mesh.vertices, mesh.faces

# 1. Nube de puntos — muestra aleatoria de la superficie
nube = mesh.sample(2048)

# 2. Voxel 32x32x32 — rejilla regular como en Pix2Vox++
pitch = 2.0 / 32
vox_pts = mesh.voxelized(pitch=pitch).points   # centros de voxels ocupados

print(f'Vertices: {len(v)} | Caras: {len(f)}')
print(f'Nube: {nube.shape}')
print(f'Voxels ocupados: {len(vox_pts)} de {32**3} posibles')

In [ ]:
# CELDA 4 — generar y guardar la figura
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CAM  = dict(eye=dict(x=1.6, y=1.0, z=0.8))
RANGO = [-1.1, 1.1]

def escena(titulo, color_titulo):
    return dict(
        camera=CAM,
        xaxis=dict(range=RANGO, visible=False),
        yaxis=dict(range=RANGO, visible=False),
        zaxis=dict(range=RANGO, visible=False),
        aspectmode='cube',
        bgcolor='white'
    )

TITULOS = [
    'Malla (.ply)',
    'Nube de puntos (.npy)',
    'Voxel 32x32x32 (binvox)',
    'STL (.stl)'
]
COLORES = ['#2a6fba', '#c0392b', '#27ae60', '#d35400']

fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'scene'}] * 4],
    subplot_titles=TITULOS,
    horizontal_spacing=0.02
)

# ----- Panel 1: Malla — superficie translucida con flatshading (muestra caras triangulares) -----
fig.add_trace(go.Mesh3d(
    x=v[:,0], y=v[:,1], z=v[:,2],
    i=f[:,0], j=f[:,1], k=f[:,2],
    color='#2a6fba', opacity=0.55,
    flatshading=True,
    lighting=dict(ambient=0.6, diffuse=0.8, specular=0.1),
    lightposition=dict(x=2, y=2, z=2)
), row=1, col=1)

# ----- Panel 2: Nube de puntos — puntos coloreados por altura -----
fig.add_trace(go.Scatter3d(
    x=nube[:,0], y=nube[:,1], z=nube[:,2],
    mode='markers',
    marker=dict(size=2.5, color=nube[:,2], colorscale='Reds', opacity=0.9)
), row=1, col=2)

# ----- Panel 3: Voxel — cuadrados en rejilla regular 32x32x32 -----
fig.add_trace(go.Scatter3d(
    x=vox_pts[:,0], y=vox_pts[:,1], z=vox_pts[:,2],
    mode='markers',
    marker=dict(size=5, color='#27ae60', symbol='square', opacity=0.55)
), row=1, col=3)

# ----- Panel 4: STL — superficie solida con sombreado realista -----
fig.add_trace(go.Mesh3d(
    x=v[:,0], y=v[:,1], z=v[:,2],
    i=f[:,0], j=f[:,1], k=f[:,2],
    color='#d35400', opacity=1.0,
    flatshading=False,
    lighting=dict(ambient=0.4, diffuse=0.9, specular=0.4, roughness=0.4, fresnel=0.2),
    lightposition=dict(x=2, y=3, z=3)
), row=1, col=4)

# Escenas
for i in range(1, 5):
    key = 'scene' if i == 1 else f'scene{i}'
    fig.update_layout(**{key: escena(TITULOS[i-1], COLORES[i-1])})

# Estilo títulos de subplots
for i, (ann, color) in enumerate(zip(fig.layout.annotations, COLORES)):
    ann.font = dict(size=16, color=color, family='Arial Black')

fig.update_layout(
    title=dict(
        text='El mismo objeto en las cuatro representaciones empleadas en el pipeline',
        font=dict(size=17, family='Arial', color='#222222'),
        x=0.5, xanchor='center'
    ),
    paper_bgcolor='white',
    height=520, width=1800,
    showlegend=False,
    margin=dict(l=5, r=5, t=90, b=5)
)

fig.write_image('figura_representaciones.png', scale=2)
print('Guardado: figura_representaciones.png')
fig.show()

## Descargar

En el panel de archivos (icono carpeta izquierda), clic derecho sobre `figura_representaciones.png` → **Descargar**.

Luego cópiala a `Latex/imagenes/figura_representaciones.png` y commitea.